# FXAssist Lite: LoRA fine-tuning (Kaggle T4)

A larger open model (**Qwen2.5-7B-Instruct-AWQ**, Apache-2.0) answers questions written from the corpus **through the real agent**; only answers that pass the agent's own citation and number checks become training data. Then a time-limited **LoRA** fine-tune of **Qwen2.5-3B-Instruct** (float16, one T4), merged and evaluated with the same evaluation set as the base model, in the same session. ADR-026, `docs/FINETUNING.md`.

Run with `make ft-push`. About 1.5 hours of GPU time. Training data stays in scratch (it contains document excerpts); reports, logs and the LoRA adapter are downloaded.

In [ ]:
# Parameters
import os
from pathlib import Path

REPO_URL = "https://github.com/<you>/fxassist_lite"  # your public copy of the repo
HOUR_BUDGET = 2.0     # the base-model evaluation is skipped if this runs out (GPU-06)
CHUNKS = 450          # corpus chunks the teacher writes questions from
TRAIN_MINUTES = 30    # LoRA training stops cleanly after this (FT-04)

MODE = "kaggle" if Path("/kaggle").exists() else "colab" if Path("/content").exists() else "mock"
print("mode:", MODE)

In [ ]:
# Get the code: a Kaggle dataset named fxassist-lite if attached, otherwise git clone.
import subprocess, sys
scratch = Path("/kaggle/tmp" if MODE == "kaggle" else "/content/scratch" if MODE == "colab" else ".lab-scratch")
scratch.mkdir(parents=True, exist_ok=True)
dataset = Path("/kaggle/input/fxassist-lite")
REPO_DIR = dataset if dataset.exists() else scratch / "fxassist_lite"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "pyyaml"], check=True)
print("code at", REPO_DIR)

In [ ]:
# Hugging Face token from notebook secrets, if set (GPU-11). Never printed, never written to a file.
try:
    if MODE == "kaggle":
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    elif MODE == "colab":
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN: set")
except Exception:
    print("HF_TOKEN: not set (fine: the models are not gated)")

## 1. Environment checks

In [ ]:
from bench import finetune, lab
cfg = lab.LabConfig.for_mode(MODE, hour_budget=HOUR_BUDGET)
env = lab.check_environment(cfg)
print({k: env[k] for k in ("gpus", "bf16_supported", "internet", "scratch_free_gb")})
if env["problems"]:
    raise SystemExit("Fix first:\n- " + "\n- ".join(env["problems"]))
lab.install_vllm(cfg)

## 2. Training data: teacher + real agent + validator (FT-01, FT-02, FT-03)

In [ ]:
finetune.make_data(cfg, REPO_DIR, CHUNKS)

## 3. LoRA fine-tune, then merge (FT-04)

In [ ]:
finetune.train(cfg, REPO_DIR, TRAIN_MINUTES)

## 4. Evaluation: fine-tuned model, then the base model, same session

In [ ]:
for row in finetune.evaluate(cfg, REPO_DIR):
    print(row)

## 5. Package

In [ ]:
zip_path = lab.package(cfg)
print("Download:", zip_path, f"({cfg.hours_left:.2f} budget hours left)")